In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

vikranthkanumuru_naruto_hand_sign_dataset_path = kagglehub.dataset_download('vikranthkanumuru/naruto-hand-sign-dataset')
wassimsghaier_frames_dataset_path = kagglehub.dataset_download('wassimsghaier/frames-dataset')
wassimsghaier_github_data_path = kagglehub.dataset_download('wassimsghaier/github-data')

print('Data source import complete.')


In [ ]:
!pip install ultralytics


In [ ]:
import os
import shutil

# === 1️⃣ CONFIGURATION ===
dataset_path = "/root/.cache/kagglehub/datasets/vikranthkanumuru/naruto-hand-sign-dataset/versions/5/Pure Naruto Hand Sign Data"
labels_input_path = "/root/.cache/kagglehub/datasets/wassimsghaier/github-data/versions/1/labels/labels/obj_train_data/Pure Naruto Hand Sign Data"

working_dataset = "/kaggle/working/fixed_dataset"
working_labels = "/kaggle/working/labels_dataset"

rename_map = {
    "hare": "rabbit",
    "ram": "sheep"
}

# === 2️⃣ UTILITY FUNCTIONS ===
def safe_reset_dir(dst):
    """Remove and recreate a directory."""
    if os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(dst, exist_ok=True)

def copy_train_test_only(src, dst):
    """
    Copy ONLY 'train' and 'test' folders (and their contents)
    from the source directory to the destination.
    Ignores any files/folders outside train/test.
    """
    safe_reset_dir(dst)
    copied = []

    for subfolder in ["train", "test"]:
        src_sub = os.path.join(src, subfolder)
        dst_sub = os.path.join(dst, subfolder)
        if os.path.exists(src_sub) and os.path.isdir(src_sub):
            shutil.copytree(src_sub, dst_sub)
            copied.append(subfolder)

    if copied:
        print(f"✅ Copied folders: {', '.join(copied)} from {src}")
    else:
        print(f"⚠️ No 'train' or 'test' folders found in {src}")

def remove_unwanted_items(base_path):
    """Remove 'zero' folders and one bad image from hare."""
    for split in ["train", "test"]:
        zero_dir = os.path.join(base_path, split, "zero")
        if os.path.exists(zero_dir):
            shutil.rmtree(zero_dir)
            print(f"🗑️ Removed folder: {zero_dir}")

    bad_image = os.path.join(base_path, "train", "hare", "IMG_e8833c4d-547b-11ea-80f8-48f17fc25591.png")
    if os.path.exists(bad_image):
        os.remove(bad_image)
        print(f"🗑️ Removed bad image: {bad_image}")

def rename_classes(base_path, rename_rules):
    """Rename folders and file names matching class names."""
    for root, dirs, files in os.walk(base_path):
        if os.path.basename(root).lower() in ["train", "test"]:
            for old_name, new_name in rename_rules.items():
                old_folder = os.path.join(root, old_name)
                new_folder = os.path.join(root, new_name)
                if os.path.exists(old_folder):
                    # Rename files inside
                    for fname in os.listdir(old_folder):
                        old_path = os.path.join(old_folder, fname)
                        new_fname = fname.replace(old_name, new_name)
                        os.rename(old_path, os.path.join(old_folder, new_fname))
                    # Rename folder itself
                    os.rename(old_folder, new_folder)
                    print(f"✅ Renamed folder: {old_name} → {new_name}")

# === 3️⃣ MAIN WORKFLOW ===
print("📂 Copying datasets (train/test only)...")
copy_train_test_only(dataset_path, working_dataset)
copy_train_test_only(labels_input_path, working_labels)

print("\n🧹 Cleaning dataset...")
remove_unwanted_items(working_dataset)

print("\n🔤 Renaming class folders and files...")
rename_classes(working_dataset, rename_map)
rename_classes(working_labels, rename_map)





In [ ]:
import os

# Path to your cleaned labels dataset
labels_dataset = "/kaggle/working/labels_dataset"

# English class name → YOLO class index
class_map = {
    "monkey": 0,
    "dragon": 1,
    "rat": 2,
    "bird": 3,
    "snake": 4,
    "ox": 5,
    "dog": 6,
    "horse": 7,
    "tiger": 8,
    "boar": 9,
    "sheep": 10,
    "rabbit": 11
}

def fix_labels_by_folder(base_path):
    """
    For each .txt file in the dataset, update the YOLO class number
    according to the folder name.
    """
    for split in ["train", "test"]:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            print(f"⚠️ Missing folder: {split_path}")
            continue

        for class_folder in os.listdir(split_path):
            folder_path = os.path.join(split_path, class_folder)
            if not os.path.isdir(folder_path):
                continue

            folder_name = class_folder.lower()
            if folder_name not in class_map:
                print(f"⚠️ Unknown class folder '{folder_name}', skipping.")
                continue

            correct_id = str(class_map[folder_name])

            for fname in os.listdir(folder_path):
                if not fname.endswith(".txt"):
                    continue

                fpath = os.path.join(folder_path, fname)
                with open(fpath, "r") as f:
                    lines = f.readlines()

                fixed_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    parts[0] = correct_id  # replace old class ID
                    fixed_lines.append(" ".join(parts))

                with open(fpath, "w") as f:
                    f.write("\n".join(fixed_lines))

            print(f"✅ Updated all labels in '{class_folder}' → class {correct_id}")

# Run it
fix_labels_by_folder(labels_dataset)

print("\n🎯 All YOLO .txt labels updated based on English folder names!")

In [11]:
def reset_path(path):
  # Reset the output directory
  if os.path.exists(path):
      shutil.rmtree(path)


In [ ]:
import shutil
import os

# Define your source paths
frames_path = "/root/.cache/kagglehub/datasets/wassimsghaier/frames-dataset/versions/1/frames_dataset"
frames_labels_path = "/root/.cache/kagglehub/datasets/wassimsghaier/frames-dataset/versions/1/labes_from_videos/obj_train_data"
github_data = "/root/.cache/kagglehub/datasets/wassimsghaier/github-data/versions/1/data"

# Destination (working) directory
working_dir = "/kaggle/working"

# Function to copy entire folder recursively
def copy_folder(src, dst):
    dst_path = os.path.join(dst, os.path.basename(src))
    if os.path.exists(dst_path):
        print(f"⚠️  Skipping '{src}' because '{dst_path}' already exists.")
    else:
        shutil.copytree(src, dst_path)
        print(f"✅ Copied '{src}' → '{dst_path}'")

# Copy all three folders
reset_path(os.path.join(working_dir,'frames_dataset'))
reset_path(os.path.join(working_dir,'obj_train_data'))
reset_path(os.path.join(working_dir,'data'))
copy_folder(frames_path, working_dir)
copy_folder(frames_labels_path, working_dir)
copy_folder(github_data, working_dir)

print("🎉 All folders and files have been copied to the working directory!")


In [14]:
working_frames_path = "/kaggle/working/frames_dataset"
working_frames_labels_path = "/kaggle/working/obj_train_data"
working_github_data = "/kaggle/working/data"
class_map = {
    "monkey": 0,
    "dragon": 1,
    "rat": 2,
    "bird": 3,
    "snake": 4,
    "ox": 5,
    "dog": 6,
    "horse": 7,
    "tiger": 8,
    "boar": 9,
    "sheep": 10,
    "rabbit": 11,
    "rabit": 11 # we have other folder have this name
}

In [ ]:


def update_txt_files(labeles_path,class_map):

    for class_folder in os.listdir(labeles_path):
        folder_path = os.path.join(labeles_path, class_folder)
        if not os.path.isdir(folder_path):
            continue

        folder_name = class_folder.lower()
        if folder_name not in class_map:
            print(f"⚠️ Unknown class folder '{folder_name}', skipping.")
            continue

        correct_id = str(class_map[folder_name])

        for fname in os.listdir(folder_path):
            if not fname.endswith(".txt"):
                continue

            fpath = os.path.join(folder_path, fname)
            with open(fpath, "r") as f:
                lines = f.readlines()

            fixed_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue
                parts[0] = correct_id  # replace old class ID
                fixed_lines.append(" ".join(parts))

            with open(fpath, "w") as f:
                f.write("\n".join(fixed_lines))

        print(f"✅ Updated all labels in '{class_folder}' → class {correct_id}")

# Run it
update_txt_files(working_frames_labels_path,class_map)



In [17]:
import os

# Paths to process
working_frames_path = "/kaggle/working/frames_dataset"
working_frames_labels_path = "/kaggle/working/obj_train_data"
kaggle_working_images="/kaggle/working/fixed_dataset"

def rename_files_in_subfolders(base_path):
    """
    Renames all files inside subfolders by prefixing them with the folder name.
    Example: frame001.jpg -> foldername_frame001.jpg
    """
    for root, dirs, files in os.walk(base_path):
        for folder in dirs:
            folder_path = os.path.join(root, folder)
            for file_name in os.listdir(folder_path):
                old_path = os.path.join(folder_path, file_name)

                if os.path.isfile(old_path):
                    new_name = f"{folder}_{file_name}"
                    new_path = os.path.join(folder_path, new_name)

                    # Avoid overwriting existing files
                    if not os.path.exists(new_path):
                        os.rename(old_path, new_path)
                        # print(f"✅ Renamed: {file_name} → {new_name}")
                    else:
                        print(f"⚠️ Skipped (already exists): {new_name}")

# Run on both directories
rename_files_in_subfolders(working_frames_path)
rename_files_in_subfolders(working_frames_labels_path)
rename_files_in_subfolders(kaggle_working_images)

print("🎉 All file names have been updated successfully!")


🎉 All file names have been updated successfully!


In [18]:
import os
import shutil

# Step 1: Create destination structure
base_dir = "/kaggle/working/all_data"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

print(f"✅ Created folder structure:\n{base_dir}\n ├── images/\n └── labels/\n")

# Step 2: Define enhanced function
def copy_selected_files(src_folder, dst_folder):
    """
    Recursively copies only image and txt files from all subfolders
    inside `src_folder` into `dst_folder`.

    - Avoids overwriting using an in-memory counter (dict)
    - Prints skipped file extensions and their paths
    """
    # Supported extensions
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}
    text_exts = {".txt"}

    # Tracking
    skipped_exts = set()
    skipped_files = []
    filename_counts = {}  # store counts to avoid overwriting

    for root, _, files in os.walk(src_folder):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            src_file_path = os.path.join(root, file)

            if ext in image_exts or ext in text_exts:
                dst_file_path = os.path.join(dst_folder, file)

                # Avoid overwriting using dict instead of while loop
                name, ext2 = os.path.splitext(file)
                if os.path.exists(dst_file_path) or name in filename_counts:
                    filename_counts[name] = filename_counts.get(name, 0) + 1
                    new_file = f"{name}_{filename_counts[name]}{ext2}"
                    dst_file_path = os.path.join(dst_folder, new_file)
                else:
                    filename_counts[name] = 0  # first occurrence

                shutil.copy2(src_file_path, dst_file_path)
                # print(f"📂 Copied: {src_file_path} → {dst_file_path}")

            else:
                skipped_exts.add(ext)
                skipped_files.append(src_file_path)

    print(f"\n🎉 Done copying from {src_folder} → {dst_folder}")

    if skipped_exts:
        print("\n⚠️ Skipped file extensions:")
        for ext in sorted(skipped_exts):
            print(f"  • {ext}")
        print("\n📄 Files skipped:")
        for path in skipped_files:
            print(f"  - {path}")
    else:
        print("\n✅ No unsupported file types found!")

# Example usage:
# copy_selected_files("/kaggle/working/frames_dataset", images_dir)
# copy_selected_files("/kaggle/working/obj_train_data", labels_dir)


✅ Created folder structure:
/kaggle/working/all_data
 ├── images/
 └── labels/



In [ ]:
github_working_images="/kaggle/working/data/data/images"
github_working_labels="/kaggle/working/data/data/labels"
frames_working_images="/kaggle/working/frames_dataset"
frames_working_labels="/kaggle/working/obj_train_data"
kaggle_working_images="/kaggle/working/fixed_dataset"
kaggle_working_labels="/kaggle/working/labels_dataset"

[os.remove(os.path.join("/kaggle/working/all_data/images", f)) for f in os.listdir("/kaggle/working/all_data/images") if os.path.isfile(os.path.join("/kaggle/working/all_data/images", f))]
[os.remove(os.path.join("/kaggle/working/all_data/labels", f)) for f in os.listdir("/kaggle/working/all_data/labels") if os.path.isfile(os.path.join("/kaggle/working/all_data/labels", f))]

copy_selected_files(github_working_images, images_dir)
copy_selected_files(frames_working_images, images_dir)
copy_selected_files(kaggle_working_images, images_dir)
copy_selected_files(github_working_labels, labels_dir)
copy_selected_files(frames_working_labels, labels_dir)
copy_selected_files(kaggle_working_labels, labels_dir)


In [20]:

def count_files(path):
    """Returns the number of files in a given folder."""
    return len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])

# Count files
images_count = count_files(images_dir)
labels_count = count_files(labels_dir)

print(f"🖼️ Total image files: {images_count}")
print(f"📝 Total label files: {labels_count}")

🖼️ Total image files: 4988
📝 Total label files: 4884


In [ ]:
def clean_unmatched_pairs(images_dir, labels_dir):
    """
    Ensures each image has a matching label and vice versa.
    Deletes unmatched files from both folders.
    """
    # Collect basenames (no extension)
    image_names = {os.path.splitext(f)[0] for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))}
    label_names = {os.path.splitext(f)[0] for f in os.listdir(labels_dir) if f.endswith(".txt")}

    # --- Delete images without labels ---
    deleted_images = 0
    for img_file in os.listdir(images_dir):
        img_path = os.path.join(images_dir, img_file)
        name, _ = os.path.splitext(img_file)
        if os.path.isfile(img_path) and name not in label_names:
            os.remove(img_path)
            deleted_images += 1
            print(f"🗑️ Deleted image (no label): {img_file}")

    # --- Delete labels without images ---
    deleted_labels = 0
    for lbl_file in os.listdir(labels_dir):
        lbl_path = os.path.join(labels_dir, lbl_file)
        name, _ = os.path.splitext(lbl_file)
        if os.path.isfile(lbl_path) and name not in image_names:
            os.remove(lbl_path)
            deleted_labels += 1
            print(f"🗑️ Deleted label (no image): {lbl_file}")

    print("\n✅ Cleanup complete!")
    print(f"🖼️ Deleted images without labels: {deleted_images}")
    print(f"📝 Deleted labels without images: {deleted_labels}")
    print(f"📦 Remaining images: {len(os.listdir(images_dir))}")
    print(f"📦 Remaining labels: {len(os.listdir(labels_dir))}")

# Run cleanup
clean_unmatched_pairs(images_dir, labels_dir)

In [22]:
import os
import shutil
import random

# ----------------------------
# Paths
# ----------------------------
base_dir = "/kaggle/working/all_data"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

yolo_base = "/kaggle/working/yolo_data"  # <-- new YOLO folder
splits = ["train", "val", "test"]
split_ratios = {"train": 0.7, "val": 0.15, "test": 0.15}
seed = 42

# ----------------------------
# 1️⃣ Delete old train/val/test folders if exist in yolo_data
# ----------------------------
if os.path.exists(yolo_base):
    shutil.rmtree(yolo_base)
    print(f"🗑️ Deleted existing folder: {yolo_base}")

for split in splits:
    os.makedirs(os.path.join(yolo_base, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, split, "labels"), exist_ok=True)

# ----------------------------
# 2️⃣ Get all image files
# ----------------------------
all_images = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]
all_images.sort()
random.seed(seed)
random.shuffle(all_images)

# ----------------------------
# 3️⃣ Split files
# ----------------------------
num_images = len(all_images)
train_end = int(split_ratios["train"] * num_images)
val_end = train_end + int(split_ratios["val"] * num_images)

split_files = {
    "train": all_images[:train_end],
    "val": all_images[train_end:val_end],
    "test": all_images[val_end:]
}

# ----------------------------
# 4️⃣ Copy files to YOLO folders
# ----------------------------
for split in splits:
    for img_file in split_files[split]:
        # Copy image
        shutil.copy2(os.path.join(images_dir, img_file),
                     os.path.join(yolo_base, split, "images", img_file))
        # Copy corresponding label
        label_file = os.path.splitext(img_file)[0] + ".txt"
        shutil.copy2(os.path.join(labels_dir, label_file),
                     os.path.join(yolo_base, split, "labels", label_file))
    print(f"✅ {split.upper()} files copied: {len(split_files[split])}")

# ----------------------------
# 5️⃣ Create YOLOv8 .yaml config
# ----------------------------
yaml_content = f"""
train: {os.path.join(yolo_base, 'train/images')}
val: {os.path.join(yolo_base, 'val/images')}
test: {os.path.join(yolo_base, 'test/images')}

nc: 1
names: ['object']
"""

yaml_path = os.path.join(yolo_base, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"📝 YOLOv8 dataset config created at: {yaml_path}")


✅ TRAIN files copied: 3133
✅ VAL files copied: 671
✅ TEST files copied: 672
📝 YOLOv8 dataset config created at: /kaggle/working/yolo_data/dataset.yaml


In [23]:
import yaml
import os

# Path to your existing dataset.yaml
yaml_path = '/kaggle/working/yolo_data/dataset.yaml'

# Check if the file exists
if not os.path.exists(yaml_path):
    raise FileNotFoundError(f"{yaml_path} not found")

# Load the YAML file
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update number of classes and names
data['nc'] = 12
data['names'] = ['monkey', 'dragon', 'rat', 'bird', 'snake', 'ox',
                 'dog', 'horse', 'tiger', 'boar', 'sheep', 'rabbit']

# Save back to the YAML file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print(f"✅ Updated {yaml_path} with nc=12 and new object names.")


✅ Updated /kaggle/working/yolo_data/dataset.yaml with nc=12 and new object names.


In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

# --- Configuration ---


# Replace 'path/to/your/custom_dataset' with the actual path to your dataset.
yolo_data_dir = '/kaggle/working/yolo_data'
data_yaml_path = os.path.join(yolo_data_dir, 'dataset.yaml')  # This is the .yaml generated before

# Check if data.yaml exists
if not os.path.exists(data_yaml_path):
    raise FileNotFoundError(f"data.yaml not found at {data_yaml_path}. Make sure your dataset is correct.")

# ----------------------------
# Load pre-trained YOLOv8 model
# ----------------------------
model = YOLO('yolov8n.pt')  # nano model for quick demo
print(f"Loaded model: {model.model_name}")
model = YOLO('yolov8n.pt') # Using the nano model for quick demonstration

print(f"Starting YOLOv8 training for model: {model.model_name}")
print(f"Dataset configuration file: {data_yaml_path}")

# --- Training Parameters for model.train() ---
# These parameters control various aspects of the training process.
# Parameters are typically passed as keyword arguments to the train() method.

training_args = {
    'data': data_yaml_path,       # Path to the data.yaml configuration file
    'epochs': 100,                # Number of training epochs. More epochs can lead to better performance
                                  # but also increase training time and risk of overfitting.
    'imgsz': 640,                 # Input image size (pixels). Images are resized to this dimension.
                                  # Common values: 320, 480, 640, 800, 1024.
    'batch': 16,                  # Batch size per GPU/CPU. Adjust based on your GPU memory.
                                  # Common values: 4, 8, 16, 32, 64. Use -1 for auto-batching to 60% GPU memory.
    'name': 'yolov8n_custom_dataset_train', # Name of the experiment. Results will be saved in 'runs/detect/name'.
    'project': 'yolov8_training_runs', # Project name where experiment runs are saved. Default is 'runs'.
    'workers': 8,                 # Number of data loading workers. Adjust based on CPU cores.
                                  # Set to 0 for Windows if you encounter issues.

    # --- Optimization Parameters ---
    'lr0': 0.01,                  # Initial learning rate. Common values: 0.01 (SGD), 0.001 (Adam).
    'lrf': 0.01,                  # Final learning rate multiplier (e.g., 0.01 means LR will end at 1% of lr0).
    'optimizer': 'AdamW',         # Optimizer to use. Options: 'SGD', 'Adam', 'AdamW', 'RMSProp'. AdamW is often recommended.
    'momentum': 0.937,            # SGD momentum (if using SGD optimizer).
    'weight_decay': 0.0005,       # Weight decay for regularization. Helps prevent overfitting.
    'warmup_epochs': 3,           # Number of warmup epochs for learning rate.
    'warmup_momentum': 0.8,       # Warmup initial momentum.
    'warmup_bias_lr': 0.1,        # Warmup initial bias learning rate.

    # --- Augmentation Parameters ---
    # These control the data augmentation applied during training to improve generalization.
    'hsv_h': 0.015,               # Hue augmentation (range 0-1).
    'hsv_s': 0.7,                 # Saturation augmentation (range 0-1).
    'hsv_v': 0.4,                 # Value (brightness) augmentation (range 0-1).
    'degrees': 0.2,               # Random rotation degrees (-degrees to +degrees).
    'translate': 0.1,             # Random translation (fraction of image size).
    'scale': 0.5,                 # Random scale (fraction of original size).
    'shear': 0.0,                 # Random shear (degrees).
    'perspective': 0.0,           # Random perspective (fraction of image size).
    'flipud': 0.0,                # Probability of flipping image upside down.
    'fliplr': 0.5,                # Probability of flipping image left-right.
    'mosaic': 1.0,                # Probability of using mosaic augmentation (combining 4 images).
                                  # Set to 0.0 to disable. Often beneficial for small objects.
    'mixup': 0.0,                 # Probability of using mixup augmentation.
    'copy_paste': 0.0,            # Probability of using copy-paste augmentation (for segmentation, can be used for detection).
    # 'auto_augment': 'randaugment', # Auto-augmentation policy. Other options: 'autoaugment', 'augmix'.

    # --- Advanced / Control Parameters ---
    'patience': 20,               # Early stopping patience (epochs without improvement on validation metrics).
                                  # Stops training if mAP@0.5-0.95 doesn't improve for this many epochs.
    'save': True,                 # Save checkpoints.
    'save_period': -1,            # Save model every 'save_period' epochs (-1 for saving only best/last).
    'val': True,                  # Perform validation during training.
    'cache': False,               # Cache images for faster training ('ram', 'disk', or False).
                                  # 'ram' is fastest but requires a lot of RAM. 'disk' caches to disk.
    'device': '0' ,
                                  # Automatically detects GPU if available.
    'pretrained': True,           # Use a pre-trained model. Set to False to train from scratch.
    'seed': 42,                   # Random seed for reproducibility.
    'plots': True,                # Save plots (e.g., confusion matrix, PR curve).
    'rect': False,                # Rectangular training. If True, images are resized to rectangles.
    'exist_ok': False,            # Overwrite existing results in `runs/detect/name` directory.
    'resume': False,              # Resume training from the last checkpoint. Set to 'True' or a specific path.
    # 'dropout': 0.0,             # Dropout regularization (for classification, not typically for YOLO detection heads).
    # 'cls': 0.5,                 # Class loss gain.
    # 'box': 7.5,                 # Box loss gain.
    # 'dfl': 1.5,                 # DFL loss gain.
    # 'pose': 12.0,               # Pose loss gain (for pose estimation models).
    # 'kpt': 2.0,                 # Keypoint loss gain (for pose estimation models).
    # 'max_det': 300,             # Maximum number of detections per image.
    # 'conf': 0.25,               # Object confidence threshold for NMS.
    # 'iou': 0.7,                 # IoU threshold for NMS.
}

# --- Train the model ---
# The model.train() method encapsulates the entire training loop.
# It handles data loading, augmentation, forward/backward passes, optimization,
# validation, metric calculation, and saving checkpoints and plots.
print("\nStarting training with the following parameters:")
for k, v in training_args.items():
    print(f"  {k}: {v}")

results = model.train(**training_args)

print("\nTraining completed!")

# --- Accessing Training Results and Metrics ---
# The 'results' object contains detailed information about the training run.
# It's an instance of `ultralytics.engine.results.Results`
# You can access various metrics and information directly from it.

# Access the best epoch's metrics
best_metrics = results.metrics # Accesses metrics from the best.pt (best performing model)
print(f"\n--- Best Model Metrics (from {results.save_dir}/weights/best.pt) ---")
print(f"Validation mAP50: {best_metrics['metrics/mAP50(B)'][0]:.4f}") # mAP at IoU=0.5
print(f"Validation mAP50-95: {best_metrics['metrics/mAP50-95(B)'][0]:.4f}") # mAP averaged over IoU thresholds 0.5 to 0.95
print(f"Validation Precision: {best_metrics['metrics/precision(B)'][0]:.4f}")
print(f"Validation Recall: {best_metrics['metrics/recall(B)'][0]:.4f}")

# You can also access training and validation loss values over epochs
# These are typically stored in the 'results.csv' or 'results.yaml' file in the run directory.
# For programmatic access, you'd usually load these files.
# The `results` object directly provides paths to these files.
print(f"\nTraining logs and checkpoints saved to: {model.trainer.save_dir}")



run_dir = model.trainer.save_dir
plots_dir = os.path.join(run_dir, 'plots')

print(f"\nAutomatically generated plots are saved in: {plots_dir}")

# Example of how to display a plot programmatically using matplotlib
def display_plot(plot_path, title):
    if os.path.exists(plot_path):
        img = cv2.imread(plot_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.title(title)
            plt.axis('off')
            plt.show()
        else:
            print(f"Warning: Could not read image at {plot_path}")
    else:
        print(f"Warning: Plot not found at {plot_path}")

print("\nAttempting to display some plots (if generated and found)...")
display_plot(os.path.join(plots_dir, 'results.png'), 'Training and Validation Metrics Over Epochs')
display_plot(os.path.join(plots_dir, 'confusion_matrix.png'), 'Confusion Matrix')
display_plot(os.path.join(plots_dir, 'F1_curve.png'), 'F1 Curve')

# --- Performing Inference with the Best Trained Model ---
# After training, you can load the best weights and perform inference.
best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
if os.path.exists(best_model_path):
    print(f"\nLoading best model from: {best_model_path}")
    trained_model = YOLO(best_model_path)

    # Example inference on a sample image (replace with your test image)
    # Ensure you have a 'test_image.jpg' in your dataset_root or provide a full path.
    sample_image_path = os.path.join(dataset_root, 'test', 'images', 'test_image.jpg') # Adjust path
    if os.path.exists(sample_image_path):
        print(f"Performing inference on: {sample_image_path}")
        inference_results = trained_model(sample_image_path)

        # Show results (this will open a window with the detected objects)
        for r in inference_results:
            r.show() # Displays image with bounding boxes
            # r.save(filename='inference_output.jpg') # Optionally save the result
    else:
        print(f"Warning: Sample inference image not found at {sample_image_path}. Skipping inference example.")
else:
    print(f"Warning: Best model weights not found at {best_model_path}. Skipping inference example.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loaded model: yolov8n.pt
Starting YOLOv8 training for model: yolov8n.pt
Dataset configuration file: /kaggle/working/yolo_data/dataset.yaml

Starting training with the following parameters:
  data: /kaggle/working/yolo_data/dataset.yaml
  epochs: 100
  imgsz: 640
  batch: 16
  name: yolov8n_custom_dataset_train
  project: yolov8_training_runs
  workers: 8
  lr0: 0.01
  lrf: 0.01
  optimizer: AdamW
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  degrees: 0.2
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  mosaic: 1.0
  mixup: 0.0
  copy_past

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
import yaml
from ultralytics import YOLO
import numpy as np
import random # For selecting random test images

# --- Configuration (using values from your original script) ---
yolo_data_dir = '/kaggle/working/yolo_data' # Your dataset root
data_yaml_path = os.path.join(yolo_data_dir, 'dataset.yaml')

# --- 1. Access and Display Best Model Metrics ---
# Since you have the 'results' object, we can directly access the metrics.
# These metrics correspond to the best performing model saved during training,
# up to the point of interruption.

print(f"\n--- Best Model Metrics (from {results.save_dir}/weights/best.pt) ---")
print(f"Validation mAP50: {results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"Validation mAP50-95: {results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"Validation Precision: {results.results_dict['metrics/precision(B)']:.4f}")
print(f"Validation Recall: {results.results_dict['metrics/recall(B)']:.4f}")
print(f"Overall Fitness Score: {results.fitness:.4f}")

# You can also print per-class mAP if needed
print("\nPer-class mAP (mAP50-95):")
for i, mAP_val in enumerate(results.maps):
    print(f"  Class {i} ({results.names[i]}): {mAP_val:.4f}")

# --- 2. Plot the Metrics and Confusion Matrix ---
# The plots are automatically saved by Ultralytics during training.
# We'll use the 'save_dir' from the 'results' object to find them.

run_dir = str(results.save_dir) # Convert PosixPath to string
plots_dir = os.path.join(run_dir, 'plots')

print(f"\nAutomatically generated plots are saved in: {plots_dir}")

def display_plot(plot_path, title):
    if os.path.exists(plot_path):
        img = cv2.imread(plot_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.title(title)
            plt.axis('off')
            plt.show()
        else:
            print(f"Warning: Could not read image at {plot_path}")
    else:
        print(f"Warning: Plot not found at {plot_path}")

print("\nAttempting to display some plots (if generated and found)...")
display_plot(os.path.join(plots_dir, 'results.png'), 'Training and Validation Metrics Over Epochs')
display_plot(os.path.join(plots_dir, 'confusion_matrix.png'), 'Confusion Matrix')
display_plot(os.path.join(plots_dir, 'F1_curve.png'), 'F1 Curve')
display_plot(os.path.join(plots_dir, 'P_curve.png'), 'Precision-Confidence Curve')
display_plot(os.path.join(plots_dir, 'R_curve.png'), 'Recall-Confidence Curve')


# --- 3. Check if Best Model Weight Exists and Use it for Prediction ---

best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
trained_model = None

if os.path.exists(best_model_path):
    print(f"\nLoading best model from: {best_model_path}")
    trained_model = YOLO(best_model_path)
    # Class names can also be retrieved directly from the loaded model
    class_names = trained_model.names
    print(f"Loaded model has {len(class_names)} classes: {class_names}")
else:
    print(f"Error: Best model weights not found at {best_model_path}. Cannot perform predictions.")

# --- Helper function for IoU calculation ---
def bbox_iou(box1, box2, xyn=True):
    """
    Calculates Intersection Over Union (IoU) of two bounding boxes.
    box = [x, y, w, h] (normalized if xyn=True, else pixel coords)
    """
    if xyn: # Convert normalized to xmin, ymin, xmax, ymax
        b1_x1, b1_y1 = box1[0] - box1[2] / 2, box1[1] - box1[3] / 2
        b1_x2, b1_y2 = box1[0] + box1[2] / 2, box1[1] + box1[3] / 2
        b2_x1, b2_y1 = box2[0] - box2[2] / 2, box2[1] - box2[3] / 2
        b2_x2, b2_y2 = box2[0] + box2[2] / 2, box2[1] + box2[3] / 2
    else: # Already pixel coords
        b1_x1, b1_y1, b1_x2, b1_y2 = box1[0], box1[1], box1[0] + box1[2], box1[1] + box1[3]
        b2_x1, b2_y1, b2_x2, b2_y2 = box2[0], box2[1], box2[0] + box2[2], box2[1] + box2[3]

    # Get the coordinates of the intersection rectangle
    inter_x1 = max(b1_x1, b2_x1)
    inter_y1 = max(b1_y1, b2_y1)
    inter_x2 = min(b1_x2, b1_x2) # Fixed: should be min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2) # Fixed: should be min(b1_y2, b2_y2)

    # Corrected intersection coordinates:
    inter_x2 = min(b1_x2, b2_x2)
    inter_y2 = min(b1_y2, b2_y2)

    # Intersection area
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # Union area
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area

    return inter_area / (union_area + 1e-6) # Add epsilon to avoid division by zero

# --- 4. Plot some examples from the test data and predict with it ---
if trained_model:
    test_images_dir = None
    test_labels_dir = None

    try:
        with open(data_yaml_path, 'r') as f:
            data_config = yaml.safe_load(f)
            test_path_relative = data_config.get('test')

            if test_path_relative:
                # Resolve the 'test' path relative to the data.yaml file
                base_dir = os.path.dirname(data_yaml_path)
                test_images_dir = os.path.abspath(os.path.join(base_dir, test_path_relative))

                # Assuming labels are in a 'labels' subdirectory parallel to 'images'
                test_labels_dir = test_images_dir.replace('images', 'labels')

                if not os.path.exists(test_labels_dir):
                     print(f"Warning: Test labels directory not found at {test_labels_dir}. Predictions correctness cannot be fully verified.")
                     test_labels_dir = None
            else:
                # Fallback to a common structure if 'test' not specified in data.yaml
                test_images_dir = os.path.join(yolo_data_dir, 'test', 'images')
                test_labels_dir = os.path.join(yolo_data_dir, 'test', 'labels')
                if not os.path.exists(test_labels_dir):
                    print(f"Warning: Test labels directory not found at {test_labels_dir}. Predictions correctness cannot be fully verified.")
                    test_labels_dir = None

        if not test_images_dir or not os.path.exists(test_images_dir):
            raise FileNotFoundError(f"Test images directory not found. Please ensure your '{data_yaml_path}' defines a 'test' path or assume default structure '{yolo_data_dir}/test/images'.")

        print(f"\nUsing test images from: {test_images_dir}")
        if test_labels_dir:
            print(f"Using test labels from: {test_labels_dir}")

    except Exception as e:
        print(f"Error parsing data.yaml or finding test paths: {e}")
        test_images_dir = None

    if test_images_dir:
        test_image_files = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not test_image_files:
            print(f"No image files found in {test_images_dir}. Skipping test predictions.")
        else:
            print(f"Found {len(test_image_files)} test images. Selecting up to 30 random images.")
            random.shuffle(test_image_files) # Shuffle to get different images each run
            images_to_predict = test_image_files[:min(30, len(test_image_files))]

            print(f"\n--- Performing predictions on {len(images_to_predict)} test images ---")
            for i, image_filename in enumerate(images_to_predict):
                image_path = os.path.join(test_images_dir, image_filename)

                print(f"\nProcessing image {i+1}/{len(images_to_predict)}: {image_filename}")

                # Load ground truth labels
                ground_truth_data = [] # List of (class_id, [x_center, y_center, width, height])
                if test_labels_dir:
                    label_filename = os.path.splitext(image_filename)[0] + '.txt'
                    label_path = os.path.join(test_labels_dir, label_filename)
                    if os.path.exists(label_path):
                        with open(label_path, 'r') as f:
                            for line in f:
                                parts = list(map(float, line.strip().split()))
                                gt_class_id = int(parts[0])
                                gt_box_normalized = parts[1:] # [x_center, y_center, width, height]
                                ground_truth_data.append((gt_class_id, gt_box_normalized))
                    else:
                        print(f"  No ground truth label file found for {image_filename}")

                # Perform inference
                inference_results = trained_model(image_path, verbose=False) # Set verbose to False to suppress per-image inference output

                # Load image for plotting
                img_display = cv2.imread(image_path)
                if img_display is None:
                    print(f"  Could not load image for display: {image_path}")
                    continue
                img_display = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)
                h, w, _ = img_display.shape

                plt.figure(figsize=(12, 12))
                plt.imshow(img_display)
                ax = plt.gca()

                # Plot Ground Truth (if available)
                if ground_truth_data:
                    for gt_class_id, gt_box_normalized in ground_truth_data:
                        x_center, y_center, box_width, box_height = gt_box_normalized
                        x_min = int((x_center - box_width / 2) * w)
                        y_min = int((y_center - box_height / 2) * h)
                        box_w = int(box_width * w)
                        box_h = int(box_height * h)

                        rect = plt.Rectangle((x_min, y_min), box_w, box_h,
                                             fill=False, color='green', linewidth=2, linestyle='--')
                        ax.add_patch(rect)
                        plt.text(x_min, y_min - 5, f"GT: {class_names.get(gt_class_id, 'Unknown')}",
                                 bbox=dict(facecolor='green', alpha=0.6), color='white', fontsize=8)

                # Process and plot predictions
                if inference_results and inference_results[0].boxes:
                    pred_boxes_xywhn = inference_results[0].boxes.xywhn.cpu().numpy() # x, y, w, h normalized
                    pred_confidences = inference_results[0].boxes.conf.cpu().numpy()
                    pred_class_ids = inference_results[0].boxes.cls.cpu().numpy().astype(int)

                    if len(pred_boxes_xywhn) == 0:
                        print("  No objects detected by the model.")

                    for pb_norm, p_conf, p_cls_id in zip(pred_boxes_xywhn, pred_confidences, pred_class_ids):
                        x_center, y_center, box_width, box_height = pb_norm
                        x_min = int((x_center - box_width / 2) * w)
                        y_min = int((y_center - box_height / 2) * h)
                        box_w = int(box_width * w)
                        box_h = int(box_height * h)

                        pred_class_name = class_names.get(p_cls_id, 'Unknown')

                        # Check correctness against ground truth
                        is_correct = False
                        matched_gt_idx = -1
                        if ground_truth_data:
                            for gt_idx, (gt_class_id, gt_box_normalized) in enumerate(ground_truth_data):
                                if p_cls_id == gt_class_id: # Same class
                                    iou = bbox_iou(pb_norm, gt_box_normalized)
                                    if iou > 0.5: # Common IoU threshold for correct detection
                                        is_correct = True
                                        matched_gt_idx = gt_idx
                                        break # Found a matching ground truth box

                        color = 'red' # Default for incorrect
                        status = "INCORRECT"
                        if is_correct:
                            color = 'blue' # Correct predictions
                            status = "CORRECT"
                            # To avoid matching the same GT box multiple times
                            if matched_gt_idx != -1:
                                ground_truth_data.pop(matched_gt_idx)

                        rect = plt.Rectangle((x_min, y_min), box_w, box_h,
                                             fill=False, color=color, linewidth=2)
                        ax.add_patch(rect)
                        plt.text(x_min, y_min - 20, f"Pred: {pred_class_name} ({p_conf:.2f}) - {status}",
                                 bbox=dict(facecolor=color, alpha=0.7), color='white', fontsize=8)

                        print(f"  - Predicted: '{pred_class_name}' (Score: {p_conf:.2f}) - {status}")
                else:
                    print("  No detections by the model for this image.")

                plt.title(f"Predictions for {image_filename}")
                plt.axis('off')
                plt.show()
    else:
        print("Skipping predictions as test image directory could not be determined or is empty.")

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

# --- Configuration ---
# 1. Define your dataset path
# This should be the root directory containing your 'data.yaml' file,
# which points to your 'train', 'val', and optionally 'test' image and label folders.
# Example structure:
# my_custom_dataset/
# ├── data.yaml
# ├── train/
# │   ├── images/
# │   └── labels/
# └── val/
#     ├── images/
#     └── labels/
# (Optional)
# └── test/
#     ├── images/
#     └── labels/

# Replace 'path/to/your/custom_dataset' with the actual path to your dataset.
yolo_data_dir = '/kaggle/working/yolo_data'
data_yaml_path = os.path.join(yolo_data_dir, 'dataset.yaml')  # This is the .yaml generated before

# Check if data.yaml exists
if not os.path.exists(data_yaml_path):
    raise FileNotFoundError(f"data.yaml not found at {data_yaml_path}. Make sure your dataset is correct.")

# ----------------------------
# Load pre-trained YOLOv8 model
# ----------------------------
model = YOLO('yolo11n.pt')  # nano model for quick demo
print(f"Loaded model: {model.model_name}")

print(f"Starting YOLOv8 training for model: {model.model_name}")
print(f"Dataset configuration file: {data_yaml_path}")

# --- Training Parameters for model.train() ---
# These parameters control various aspects of the training process.
# Parameters are typically passed as keyword arguments to the train() method.

training_args = {
    'data': data_yaml_path,       # Path to the data.yaml configuration file
    'epochs': 100,                # Number of training epochs. More epochs can lead to better performance
                                  # but also increase training time and risk of overfitting.
    'imgsz': 640,                 # Input image size (pixels). Images are resized to this dimension.
                                  # Common values: 320, 480, 640, 800, 1024.
    'batch': 16,                  # Batch size per GPU/CPU. Adjust based on your GPU memory.
                                  # Common values: 4, 8, 16, 32, 64. Use -1 for auto-batching to 60% GPU memory.
    'name': 'yolo11n_custom_dataset_train', # Name of the experiment. Results will be saved in 'runs/detect/name'.
    'project': 'yolo11n_training_runs', # Project name where experiment runs are saved. Default is 'runs'.
    'workers': 8,                 # Number of data loading workers. Adjust based on CPU cores.
                                  # Set to 0 for Windows if you encounter issues.

    # --- Optimization Parameters ---
    'lr0': 0.01,                  # Initial learning rate. Common values: 0.01 (SGD), 0.001 (Adam).
    'lrf': 0.01,                  # Final learning rate multiplier (e.g., 0.01 means LR will end at 1% of lr0).
    'optimizer': 'AdamW',         # Optimizer to use. Options: 'SGD', 'Adam', 'AdamW', 'RMSProp'. AdamW is often recommended.
    'momentum': 0.937,            # SGD momentum (if using SGD optimizer).
    'weight_decay': 0.0005,       # Weight decay for regularization. Helps prevent overfitting.
    'warmup_epochs': 3,           # Number of warmup epochs for learning rate.
    'warmup_momentum': 0.8,       # Warmup initial momentum.
    'warmup_bias_lr': 0.1,        # Warmup initial bias learning rate.

    # --- Augmentation Parameters ---
    # These control the data augmentation applied during training to improve generalization.
    'hsv_h': 0.015,               # Hue augmentation (range 0-1).
    'hsv_s': 0.7,                 # Saturation augmentation (range 0-1).
    'hsv_v': 0.4,                 # Value (brightness) augmentation (range 0-1).
    'degrees': 0.2,               # Random rotation degrees (-degrees to +degrees).
    'translate': 0.1,             # Random translation (fraction of image size).
    'scale': 0.5,                 # Random scale (fraction of original size).
    'shear': 0.0,                 # Random shear (degrees).
    'perspective': 0.0,           # Random perspective (fraction of image size).
    'flipud': 0.0,                # Probability of flipping image upside down.
    'fliplr': 0.5,                # Probability of flipping image left-right.
    'mosaic': 1.0,                # Probability of using mosaic augmentation (combining 4 images).
                                  # Set to 0.0 to disable. Often beneficial for small objects.
    'mixup': 0.0,                 # Probability of using mixup augmentation.
    'copy_paste': 0.0,            # Probability of using copy-paste augmentation (for segmentation, can be used for detection).
    # 'auto_augment': 'randaugment', # Auto-augmentation policy. Other options: 'autoaugment', 'augmix'.

    # --- Advanced / Control Parameters ---
    'patience': 20,               # Early stopping patience (epochs without improvement on validation metrics).
                                  # Stops training if mAP@0.5-0.95 doesn't improve for this many epochs.
    'save': True,                 # Save checkpoints.
    'save_period': -1,            # Save model every 'save_period' epochs (-1 for saving only best/last).
    'val': True,                  # Perform validation during training.
    'cache': False,               # Cache images for faster training ('ram', 'disk', or False).
                                  # 'ram' is fastest but requires a lot of RAM. 'disk' caches to disk.
    'device': '0' ,
                                  # Automatically detects GPU if available.
    'pretrained': True,           # Use a pre-trained model. Set to False to train from scratch.
    'seed': 42,                   # Random seed for reproducibility.
    'plots': True,                # Save plots (e.g., confusion matrix, PR curve).
    'rect': False,                # Rectangular training. If True, images are resized to rectangles.
    'exist_ok': False,            # Overwrite existing results in `runs/detect/name` directory.
    'resume': False,              # Resume training from the last checkpoint. Set to 'True' or a specific path.
    # 'dropout': 0.0,             # Dropout regularization (for classification, not typically for YOLO detection heads).
    # 'cls': 0.5,                 # Class loss gain.
    # 'box': 7.5,                 # Box loss gain.
    # 'dfl': 1.5,                 # DFL loss gain.
    # 'pose': 12.0,               # Pose loss gain (for pose estimation models).
    # 'kpt': 2.0,                 # Keypoint loss gain (for pose estimation models).
    # 'max_det': 300,             # Maximum number of detections per image.
    # 'conf': 0.25,               # Object confidence threshold for NMS.
    # 'iou': 0.7,                 # IoU threshold for NMS.
}

# --- Train the model ---
# The model.train() method encapsulates the entire training loop.
# It handles data loading, augmentation, forward/backward passes, optimization,
# validation, metric calculation, and saving checkpoints and plots.
print("\nStarting training with the following parameters:")
for k, v in training_args.items():
    print(f"  {k}: {v}")

results = model.train(**training_args)

print("\nTraining completed!")

# --- Accessing Training Results and Metrics ---
# The 'results' object contains detailed information about the training run.
# It's an instance of `ultralytics.engine.results.Results`
# You can access various metrics and information directly from it.

# Access the best epoch's metrics
best_metrics = results.metrics # Accesses metrics from the best.pt (best performing model)
print(f"\n--- Best Model Metrics (from {results.save_dir}/weights/best.pt) ---")
print(f"Validation mAP50: {best_metrics['metrics/mAP50(B)'][0]:.4f}") # mAP at IoU=0.5
print(f"Validation mAP50-95: {best_metrics['metrics/mAP50-95(B)'][0]:.4f}") # mAP averaged over IoU thresholds 0.5 to 0.95
print(f"Validation Precision: {best_metrics['metrics/precision(B)'][0]:.4f}")
print(f"Validation Recall: {best_metrics['metrics/recall(B)'][0]:.4f}")

# You can also access training and validation loss values over epochs
# These are typically stored in the 'results.csv' or 'results.yaml' file in the run directory.
# For programmatic access, you'd usually load these files.
# The `results` object directly provides paths to these files.
print(f"\nTraining logs and checkpoints saved to: {model.trainer.save_dir}")

# --- Viewing Automatically Generated Plots ---
# Ultralytics automatically generates various plots during training,
# saving them to the 'plots' subfolder within your experiment's run directory
# (e.g., yolov8_training_runs/yolov8n_custom_dataset_train/plots/).

# Common plots include:
# - F1_curve.png: F1 Score vs. Confidence Threshold
# - P_curve.png: Precision vs. Confidence Threshold
# - R_curve.png: Recall vs. Confidence Threshold
# - confusion_matrix.png: Confusion matrix for object detection
# - confusion_matrix_normalized.png: Normalized confusion matrix
# - labels.jpg: Distribution of labels
# - labels_correlogram.jpg: Label correlations
# - results.png: Training/Validation metrics over epochs (mAP, P, R, losses)

run_dir = model.trainer.save_dir
plots_dir = os.path.join(run_dir, 'plots')

print(f"\nAutomatically generated plots are saved in: {plots_dir}")

# Example of how to display a plot programmatically using matplotlib
def display_plot(plot_path, title):
    if os.path.exists(plot_path):
        img = cv2.imread(plot_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.title(title)
            plt.axis('off')
            plt.show()
        else:
            print(f"Warning: Could not read image at {plot_path}")
    else:
        print(f"Warning: Plot not found at {plot_path}")

print("\nAttempting to display some plots (if generated and found)...")
display_plot(os.path.join(plots_dir, 'results.png'), 'Training and Validation Metrics Over Epochs')
display_plot(os.path.join(plots_dir, 'confusion_matrix.png'), 'Confusion Matrix')
display_plot(os.path.join(plots_dir, 'F1_curve.png'), 'F1 Curve')

# --- Performing Inference with the Best Trained Model ---
# After training, you can load the best weights and perform inference.
best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
if os.path.exists(best_model_path):
    print(f"\nLoading best model from: {best_model_path}")
    trained_model = YOLO(best_model_path)

    # Example inference on a sample image (replace with your test image)
    # Ensure you have a 'test_image.jpg' in your dataset_root or provide a full path.
    sample_image_path = os.path.join(dataset_root, 'test', 'images', 'test_image.jpg') # Adjust path
    if os.path.exists(sample_image_path):
        print(f"Performing inference on: {sample_image_path}")
        inference_results = trained_model(sample_image_path)

        # Show results (this will open a window with the detected objects)
        for r in inference_results:
            r.show() # Displays image with bounding boxes
            # r.save(filename='inference_output.jpg') # Optionally save the result
    else:
        print(f"Warning: Sample inference image not found at {sample_image_path}. Skipping inference example.")
else:
    print(f"Warning: Best model weights not found at {best_model_path}. Skipping inference example.")

Loaded model: yolo11n.pt
Starting YOLOv8 training for model: yolo11n.pt
Dataset configuration file: /kaggle/working/yolo_data/dataset.yaml

Starting training with the following parameters:
  data: /kaggle/working/yolo_data/dataset.yaml
  epochs: 100
  imgsz: 640
  batch: 16
  name: yolo11n_custom_dataset_train
  project: yolo11n_training_runs
  workers: 8
  lr0: 0.01
  lrf: 0.01
  optimizer: AdamW
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  degrees: 0.2
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  mosaic: 1.0
  mixup: 0.0
  copy_paste: 0.0
  patience: 20
  save: True
  save_period: -1
  val: True
  cache: False
  device: 0
  pretrained: True
  seed: 42
  plots: True
  rect: False
  exist_ok: False
  resume: False
New https://pypi.org/project/ultralytics/8.3.218 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.217 🚀 Py